# GPT：从 Decoder-only 原理到生成接口

> **本章定位**：在 Encoder–Decoder Transformer 基础上建立 GPT 的 Decoder-only（仅解码器）结构、因果语言模型目标与自回归生成契约。

> **章节边界**：开放权重模型的仓库审计，以及 RoPE、RMSNorm、GQA 与 MoE 等架构增量，见 `E10_open_model.ipynb`；KV Cache 的系统优化、连续批处理与推理并行见 `A50_inference_optimization.ipynb`。本章仅说明 KV Cache 的语义边界。

> **总览**：内容以小型字符语料离线验证 Token Embedding、因果自注意力、GPT Block、Next-Token Loss 和自回归解码，并通过随机初始化的 Hugging Face GPT-2 配置核对标准输入输出接口。

<!-- diagram:gpt-decoder-only-overview -->

![架构图：GPT Decoder-only 主干、因果掩码、残差块与输出形状](assets/figures/31_nlp_gpt/gpt-decoder-only-overview.svg)

[TikZ 源文件](assets/figures/31_nlp_gpt/gpt-decoder-only-overview.tex)



## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 共同基础：自回归生成 |
| 本章定位 | 在 Transformer 理论基础上集中学习 Decoder-only、Causal LM 与自回归生成。 |
| 先修知识 | 完成 `30`；理解 Attention、残差、LayerNorm 和 causal mask。 |
| 预计时间 | 90～120 分钟 |
| 运行资源 | CPU 即可；标准库部分使用随机初始化小模型。 |
| 输入 | 单序列 token ID、causal mask 与 next-token labels。 |
| 交付物 | 最小 GPT、训练与生成闭环及 Hugging Face GPT 输入输出契约。 |

### 1.1．学习目标

完成本章后，读者能够解释 Decoder-only 架构与因果语言模型目标，实现并验证 GPT Block、训练闭环和自回归解码，将原理对象映射到 Transformers 的 GPT 接口，并说明 KV Cache 的正确性与状态边界。


### 1.2．环境与依赖

输入是整数 token id，输出是每个位置对词表的 logits。代码自动选择 CUDA、Apple Silicon MPS 或 CPU。


In [ ]:
# 导入依赖、固定随机种子并选择可用设备。

import math
import random
import torch
from torch import nn
import torch.nn.functional as F

# 固定初始化与训练抽样；质量比较需使用预先登记的多个种子。
SEED = 42  # 仅支持实验重放，不保证跨设备或库版本逐位一致。
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print("device:", DEVICE)


## 2．直觉与输入输出契约

### 2.1．最小 Tokenizer 与训练样本

为隔离 GPT 原理，本章采用内置字符级 Tokenizer，因此能够独立运行。正式项目应通过 `AutoTokenizer.from_pretrained()` 加载与模型共同版本化的标准 Tokenizer 资产。

- 输入：长度为 `T` 的 token 序列；
- 标签：同一序列左移一位；
- 验证：输入和标签形状相同，且 `labels[:, :-1] == input_ids[:, 1:]`。


In [ ]:
# 建立字符级词表，并用滑动窗口构造输入与右移一位的标签。

CORPUS_REPETITIONS = 80  # 仅增加滑动窗口数量，不扩展数据分布，不能替代真实多样语料。
unit_text = "大模型学习需要理解因果注意力。"
# BOS/EOS 在当前资产中按顺序取得 ID 0/1；词表顺序变化时必须同步配置与 Checkpoint。
special_tokens = ["<bos>", "<eos>"]
chars = sorted(set(unit_text))
vocab_tokens = special_tokens + chars
stoi = {token: index for index, token in enumerate(vocab_tokens)}
itos = {index: ch for ch, index in stoi.items()}
vocab_size = len(vocab_tokens)
bos_token_id = stoi["<bos>"]
eos_token_id = stoi["<eos>"]

def my_encode(text):
    """把字符文本编码为当前教学词表中的 token ID 序列。"""
    return [stoi[ch] for ch in text]

def my_decode(ids, skip_special_tokens=True):
    """把 token ID 还原为文本，并可选择移除 BOS/EOS。"""
    decoded_tokens = [itos[int(index)] for index in ids]
    if skip_special_tokens:
        decoded_tokens = [token for token in decoded_tokens if token not in special_tokens]
    return "".join(decoded_tokens)

# 每次重复都保留独立 BOS/EOS 边界，使特殊 Token 进入监督序列。
training_unit = [bos_token_id, *my_encode(unit_text), eos_token_id]
tokens = torch.tensor(training_unit * CORPUS_REPETITIONS, dtype=torch.long)
block_size = 16  # 因果窗口长度；增大会显著提高 Attention 计算与位置 Embedding 占用。

# 默认 Batch 8 用于形状检查；训练另用 16，改变后需按 tokens/update 联动复核学习率。
def my_get_batch(batch_size=8):
    """随机抽取定长窗口并返回右移一位的 next-token 训练批次。"""
    # 随机选择窗口起点；标签窗口整体右移一位形成 next-token 目标。
    starts = torch.randint(0, len(tokens) - block_size - 1, (batch_size,))
    input_ids = torch.stack([tokens[i:i + block_size] for i in starts])
    labels = torch.stack([tokens[i + 1:i + block_size + 1] for i in starts])
    return input_ids.to(DEVICE), labels.to(DEVICE)

input_ids, labels = my_get_batch()
print("vocab_size:", vocab_size, "sample:", my_decode(input_ids[0].cpu()))


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

Decoder-only 模型把序列联合概率分解为一系列条件概率，并用因果 Mask 禁止位置 $t$ 读取未来 Token：

$$
p(x_{1:L})=\prod_{t=1}^{L}p(x_t\mid x_{<t}),\qquad
M_{t,s}=\begin{cases}0,&s\le t\\-\infty,&s>t\end{cases}
$$

其中，$x_t$ 是位置 $t$ 的 Token，$L$ 是序列长度，$M\in\mathbb{R}^{L\times L}$。训练时 Logits 为 $[B,L,V]$，目标是右移后的 $[B,L]$ Token ID。`MyCausalSelfAttention` 对应因果可见性与 Attention 聚合，`CrossEntropyLoss` 对应条件负对数似然。对于多头注意力，查询头数 $N_q$ 与隐藏维度需满足实现的分头约束；GQA 还区分查询头 $N_q$ 和 KV 头 $N_{kv}$。头数关系描述网络结构，是否能按张量并行切分则取决于具体 Runtime 是否支持 KV 头复制。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．因果多头自注意力

与 BERT 的双向注意力不同，GPT 在位置 $t$ 只能读取不晚于 $t$ 的 token：

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}+M_{\mathrm{causal}}\right)V$$

上三角未来位置在 softmax 前被填为负无穷，因此概率为 0。实现采用 Pre-LayerNorm（预层归一化）友好的张量形状 `[B, heads, T, head_dim]`。


In [ ]:
# 从零实现带下三角 Mask 的多头注意力，禁止当前位置读取未来 Token。
# Dropout 默认为 0，以形成可复核的原理基线；非零取值应依据验证集过拟合情况确定。

class MyCausalSelfAttention(nn.Module):
    """实现带未来位置屏蔽的多头因果自注意力。"""
    # 注册 QKV 投影、输出投影与注意力 Dropout。
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        """创建打包 QKV、输出投影及注意力 Dropout。"""
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attention=False):
        """对 `[B,T,E]` 输入执行因果注意力，并可返回逐头权重。"""
        batch, length, dim = x.shape
        # 一次线性投影后沿最后一维切出 Q、K、V，减少独立投影调用。
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def my_split_heads(tensor):
            """把模型维度重排为独立的多头表示。"""
            return tensor.view(
                batch, length, self.num_heads, self.head_dim
            ).transpose(1, 2)

        q, k, v = map(my_split_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        # 上三角位置代表未来 token，填为负无穷后 softmax 权重变为 0。
        future = torch.triu(
            torch.ones(length, length, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        scores = scores.masked_fill(future, float("-inf"))
        attention = self.dropout(F.softmax(scores, dim=-1))
        context = attention @ v
        context = context.transpose(1, 2).contiguous().view(
            batch, length, dim
        )
        output = self.output(context)
        return (output, attention) if return_attention else output

# 32 维与 4 个头形成每头 8 维的机制检查；调整后必须保持整除。
attention_layer = MyCausalSelfAttention(embed_dim=32, num_heads=4).to(DEVICE)
# 固定形状：hidden.shape = [2, 6, 32]。
hidden = torch.randn(2, 6, 32, device=DEVICE)
output, attention = attention_layer(hidden, return_attention=True)
future = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
print("causal attention shape:", attention.shape)


### 3.2．GPT Block 与最小 GPT

每个 Block 使用 `LayerNorm → Attention → 残差 → LayerNorm → FFN → 残差`。输出层与 token embedding 权重共享，可减少参数并让输入、输出处于同一语义空间。


In [ ]:
# 组合注意力、MLP、残差和归一化，堆叠为最小 GPT。

class MyGPTBlock(nn.Module):
    """组合 Pre-Norm 因果注意力、前馈网络与两条残差支路。"""
    # 组合 Pre-Norm 注意力与前馈网络。
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        """按模型宽度和头数构造一个 GPT Block。"""
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = MyCausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    # 两个子层均通过残差连接回主干。
    def forward(self, x):
        """依次执行注意力和 FFN 残差更新。"""
        x = x + self.attention(self.norm1(x))
        return x + self.ffn(self.norm2(x))


# 64 维、4 头、2 层与四倍 MLP 构成结构完整的微型 GPT；扩大任一项都会增加容量与延迟。
class MyGPT(nn.Module):
    """实现含 Token/位置 Embedding、GPT Blocks 和词表头的微型 Causal LM。"""
    def __init__(self, vocab_size, block_size, embed_dim=64, num_heads=4, num_layers=2):
        """创建指定上下文长度与深度的 GPT 参数和权重绑定。"""
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.ModuleList([
            MyGPTBlock(embed_dim, num_heads) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids, labels=None):
        """返回逐位置词表 logits，并在提供标签时计算 next-token 交叉熵。"""
        batch, length = input_ids.shape
        if length > self.block_size:
            raise ValueError(f"sequence length {length} exceeds {self.block_size}")
        positions = torch.arange(length, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1)
            )
        return {"logits": logits, "loss": loss}

model = MyGPT(vocab_size, block_size).to(DEVICE)
result = model(input_ids, labels)
print("parameters:", sum(p.numel() for p in model.parameters()))


## 4．证据验证

### 4.1．训练与自回归生成

训练使用 teacher forcing（教师强制）：一次并行预测所有下一 token。生成必须逐 token 进行；温度控制分布平滑度，`top_k` 限制候选集。生产模型还要处理停止 token、批量请求和 KV Cache。

<!-- diagram:gpt-shifted-labels -->
因果语言模型训练把同一条 token 序列错开一位作为输入与标签：

![架构图：因果语言模型输入与标签错位及 Cross-Entropy 数据流](assets/figures/31_nlp_gpt/gpt-shifted-labels.svg)

[TikZ 源文件](assets/figures/31_nlp_gpt/gpt-shifted-labels.tex)


In [ ]:
# 使用 next-token loss 训练模型，并记录损失变化。
from tqdm.auto import trange

TRAINING_STEPS = 80  # 有限更新预算只验证闭环；长周期训练需配置 Warmup 与衰减。
TRAIN_BATCH_SIZE = 16  # 每步 256 个窗口 Token；改变后需联动复核学习率与显存。
LEARNING_RATE = 3e-3  # 随机初始化微型模型的固定步长；震荡时降低，配置变化后重新搜索。
WEIGHT_DECAY = 0.01  # 当前正则起点，应在固定更新预算下比较验证指标。
ADAM_BETAS = (0.9, 0.999)  # 固定一阶与二阶矩衰减，版本化训练配置必须显式记录。
ADAM_EPSILON = 1e-8  # AdamW 分母的数值稳定项。
MAX_GRAD_NORM = 1.0  # 异常梯度保护上限；频繁裁剪时应检查学习率、Batch 与损失尺度。
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY
)
losses = []
model.train()
# 每个训练步完成一次前向、反向与参数更新。
training_progress = trange(
    TRAINING_STEPS, desc="训练最小 GPT", unit="step", dynamic_ncols=True
)
for step in training_progress:
    x, y = my_get_batch(batch_size=TRAIN_BATCH_SIZE)
    loss = model(x, y)["loss"]
    # 清空上一轮梯度，避免 PyTorch 默认的梯度累积。
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    training_progress.set_postfix(loss=f"{losses[-1]:.4f}")

print(f"loss: {losses[0]:.3f} → {losses[-1]:.3f}")


In [ ]:
# 从提示序列开始逐 token 追加预测结果，完成自回归生成。
# 默认 temperature=0.8、top-k=5 用于短文本采样；模型变化后需重评质量、多样性与安全指标。

TEMPERATURE_FLOOR = 1e-5  # 仅防止教学函数除零；生产接口应直接拒绝非正 Temperature。

@torch.no_grad()
def my_generate(
    model, input_ids, max_new_tokens, temperature=0.8, top_k=5, eos_token_id=None
):
    """使用 temperature 与 top-k 逐 Token 采样，遇到 EOS 时提前结束。"""
    model.eval()
    finished = torch.zeros(input_ids.size(0), dtype=torch.bool, device=input_ids.device)
    for _ in trange(
        max_new_tokens, desc="自回归生成", unit="token-step", dynamic_ncols=True
    ):
        context = input_ids[:, -model.block_size:]
        # 执行前向计算，得到后续损失或解码需要的模型输出。
        logits = model(context)["logits"][:, -1, :] / max(temperature, TEMPERATURE_FLOOR)
        k = min(top_k, logits.size(-1))
        values, _ = torch.topk(logits, k)
        logits[logits < values[:, [-1]]] = float("-inf")
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        if eos_token_id is not None:
            next_token = torch.where(
                finished[:, None],
                torch.full_like(next_token, eos_token_id),
                next_token,
            )
            finished |= next_token.squeeze(1).eq(eos_token_id)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if bool(finished.all()):
            break
    return input_ids

GENERATION_SEED = 42  # 独立固定解码随机序列；并发服务应为每个请求使用独立 Generator。
MAX_GENERATION_TOKENS = 24  # 足以观察短程自回归；提高会增加延迟与无依据续写风险。
torch.manual_seed(GENERATION_SEED)
# 固定形状：prompt.shape = [1, 2]。
prompt = torch.tensor(
    [[bos_token_id, *my_encode("大模型")]], dtype=torch.long, device=DEVICE
)
generated = my_generate(
    model, prompt, max_new_tokens=MAX_GENERATION_TOKENS, eos_token_id=eos_token_id
)
print(my_decode(generated[0].cpu()))


#### 4.1.1．同一因果三角形的两只时钟

**学习问题。** 因果语言模型为什么能在训练时一次并行得到所有位置的 next-token logits，生成时却必须沿已经产生的前缀逐 Token 推进？对同一条已实现序列，完整前向与逐前缀前向在对应位置上是否给出相同 logits？

本分镜直接复用上一单元训练后的 `model` 与既有 `generated` 序列，不重新采样，也不实现另一套生成算法。设可视化序列为 `T+1` 个真实 Token：完整路径把前 `T` 个 Token 一次送入模型，得到 `[1,T,V]` logits；前缀路径依次送入长度为 `1...T` 的同一前缀，每次只读取最后位置的 `[1,V]` logits。两条路径使用相同参数、位置编号和 causal mask。

运行前检查以下不变量：

- 模型切换到 `eval()`，比较过程位于 `torch.no_grad()` 中，避免 Dropout 与计算图改变证据。
- 完整路径位置 `t` 的 logits 应与长度为 `t+1` 的前缀路径末位 logits 一致；FP32 对照采用 `rtol=1e-4`、`atol=1e-5`，只吸收不同矩阵形状触发的 Kernel 舍入差异。
- 用于数值比较和绘图的 logits 统一沿 `detach() → float() → cpu()` 转换；图中 Token 来自同一份已生成序列。


In [ ]:
# 复用训练后模型与既有生成序列，对照完整前向和逐前缀前向的同位置 logits。
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLOCK_MAX_POSITIONS = 8  # 只展示短序列以保持 Token 与前缀阶梯可读；不改变模型上下文契约。
CLOCK_RTOL = 1e-4  # 允许不同序列形状触发的 FP32 Kernel 相对舍入差异。
CLOCK_ATOL = 1e-5  # 接近零的 Logit 使用绝对容差；精度或后端变化后需按误差分布复核。
clock_position_count = min(
    CLOCK_MAX_POSITIONS, generated.size(1) - 1, model.block_size
)
if clock_position_count < 2:
    raise RuntimeError("既有生成序列过短，无法形成训练与生成时钟对照")

# 多保留一个真实 Token，使每个输入位置都有同一生成路径上的 next-token 目标。
clock_sequence_ids = generated[:1, :clock_position_count + 1]
clock_input_ids = clock_sequence_ids[:, :-1]
clock_target_ids = clock_sequence_ids[:, 1:]

model.eval()
with torch.no_grad():
    clock_full_logits = model(clock_input_ids)["logits"]
    clock_prefix_last_logits = torch.stack(
        [
            model(clock_input_ids[:, :prefix_length])["logits"][:, -1, :]
            for prefix_length in range(1, clock_position_count + 1)
        ],
        dim=1,
    )

clock_full_logits_cpu = clock_full_logits.detach().float().cpu()
clock_prefix_logits_cpu = clock_prefix_last_logits.detach().float().cpu()
clock_sequence_ids_cpu = clock_sequence_ids.detach().cpu()
clock_input_ids_cpu = clock_input_ids.detach().cpu()
clock_target_ids_cpu = clock_target_ids.detach().cpu()
torch.testing.assert_close(
    clock_full_logits_cpu,
    clock_prefix_logits_cpu,
    rtol=CLOCK_RTOL,
    atol=CLOCK_ATOL,
)

clock_position_errors = (
    clock_full_logits_cpu - clock_prefix_logits_cpu
).abs().squeeze(0).amax(dim=-1)
clock_max_abs_error = float(clock_position_errors.max())
clock_top1_matches = clock_full_logits_cpu.argmax(dim=-1).eq(
    clock_prefix_logits_cpu.argmax(dim=-1)
)
clock_token_labels = [
    {"\n": "<NL>", " " : "<SP>", "\t": "<TAB>"}.get(
        itos[int(token_id)], itos[int(token_id)]
    )
    for token_id in clock_sequence_ids_cpu[0]
]

input_color = "#0072B2"
target_color = "#E69F00"
evidence_color = "#009E73"
neutral_color = "#8C8C8C"
figure, (parallel_axis, serial_axis, error_axis) = plt.subplots(
    1, 3, figsize=(16, 5.6), gridspec_kw={"width_ratios": [1.05, 1.45, 0.8]},
    constrained_layout=True,
)

# 第一帧：同一次完整前向中，所有因果位置同时形成 next-token logits。
for position in range(clock_position_count):
    parallel_axis.add_patch(Rectangle(
        (position + 0.06, 1.18), 0.88, 0.62, facecolor=input_color, alpha=0.82, edgecolor="none"
    ))
    parallel_axis.add_patch(Rectangle(
        (position + 0.06, 0.12), 0.88, 0.62, facecolor=target_color, alpha=0.82, edgecolor="none"
    ))
    parallel_axis.text(
        position + 0.5, 1.49, clock_token_labels[position], ha="center", va="center", color="white", fontsize=9
    )
    parallel_axis.text(
        position + 0.5, 0.43, clock_token_labels[position + 1], ha="center", va="center", color="black", fontsize=9
    )
    parallel_axis.annotate(
        "", xy=(position + 0.5, 0.78), xytext=(position + 0.5, 1.14),
        arrowprops={"arrowstyle": "-|>", "color": neutral_color, "lw": 1.1},
    )
parallel_axis.text(-0.12, 1.49, "输入", ha="right", va="center")
parallel_axis.text(-0.12, 0.43, "目标", ha="right", va="center")
parallel_axis.set(
    title="① 训练时钟：一次前向覆盖全部位置",
    xlim=(-0.75, clock_position_count + 0.05), ylim=(-0.12, 2.08),
)
parallel_axis.axis("off")

# 第二帧：沿同一条已生成路径逐行增加前缀，每个时钟只产生一个新位置。
for step in range(1, clock_position_count + 1):
    row = step - 1
    for position in range(step):
        serial_axis.add_patch(Rectangle(
            (position + 0.05, row + 0.1), 0.9, 0.72,
            facecolor=input_color, alpha=0.82, edgecolor="none",
        ))
        serial_axis.text(
            position + 0.5, row + 0.46, clock_token_labels[position],
            ha="center", va="center", color="white", fontsize=8,
        )
    serial_axis.add_patch(Rectangle(
        (step + 0.05, row + 0.1), 0.9, 0.72,
        facecolor=target_color, alpha=0.82, edgecolor="none",
    ))
    serial_axis.text(
        step + 0.5, row + 0.46, clock_token_labels[step],
        ha="center", va="center", color="black", fontsize=8,
    )
    serial_axis.annotate(
        "", xy=(step + 0.02, row + 0.46), xytext=(step - 0.02, row + 0.46),
        arrowprops={"arrowstyle": "-|>", "color": neutral_color, "lw": 1.0},
    )
    serial_axis.text(-0.18, row + 0.46, f"时钟 {step}", ha="right", va="center", fontsize=8)
serial_axis.set(
    title="② 生成时钟：前缀逐步增长",
    xlim=(-1.05, clock_position_count + 1.05), ylim=(-0.15, clock_position_count + 0.15),
)
serial_axis.invert_yaxis()
serial_axis.axis("off")

# 第三帧：完整前向与逐前缀前向在每个对应位置上给出相同 logits。
positions = torch.arange(clock_position_count).numpy()
error_values = clock_position_errors.numpy()
error_axis.bar(positions, error_values, color=evidence_color, alpha=0.85)
error_ceiling = max(clock_max_abs_error * 1.25, CLOCK_ATOL * 1.2)
error_axis.set(
    title="③ 同位置 Logit 等价性", xlabel="输入位置 t", ylabel="最大绝对误差",
    xticks=positions, ylim=(0.0, error_ceiling),
)
error_axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
error_axis.grid(axis="y", alpha=0.25)
error_axis.text(
    0.03, 0.97,
    f"max={clock_max_abs_error:.2e}\ntop-1 一致={int(clock_top1_matches.sum())}/{clock_position_count}",
    transform=error_axis.transAxes, ha="left", va="top",
)
plt.show()

print({
    "input_shape": tuple(clock_input_ids_cpu.shape),
    "target_shape": tuple(clock_target_ids_cpu.shape),
    "full_logits_shape": tuple(clock_full_logits_cpu.shape),
    "prefix_last_logits_shape": tuple(clock_prefix_logits_cpu.shape),
    "full_forward_calls": 1,
    "prefix_probe_forward_calls": clock_position_count,
    "max_abs_logit_error": clock_max_abs_error,
    "top1_matches": f"{int(clock_top1_matches.sum())}/{clock_position_count}",
})


**应观察到的结论。** 第一帧中，完整序列的 `T` 个位置在一次模型调用内形成 `T` 组 next-token logits；第二帧中，同一条已生成路径必须先确定较短前缀，才能进入下一个时钟；第三帧及数值摘要表明，完整前向位置 `t` 与长度为 `t+1` 的前缀前向末位 logits 在浮点容差内一致。由此可见，训练的并行性来自已知目标序列和 causal mask 允许同时计算各位置，而不是允许当前位置读取未来 Token。

**不可误读的边界。** “一次并行”描述同一层内多个序列位置可以批量计算，网络层之间仍有先后依赖。逐前缀调用只用于核对同一已实现序列的条件分布，没有重新执行 Temperature、Top-k 或随机采样，也没有证明某条生成路径代表完整概率分布。生产推理通常使用 KV Cache 避免重复计算历史 K/V，但下一个 Token 仍依赖已经确定的前缀；相关系统优化在 `A50_inference_optimization.ipynb` 展开。当前微型模型仅验证因果计算契约，不能据此判断语言质量。


### 4.2．因果注意力热力图：信息流边界是否成立

**学习问题。** causal mask 是否在真实注意力权重中完全切断未来位置，同时允许不同注意力头在合法历史范围内形成不同分配？

本图不构造理想化矩阵，而是把当前 `input_ids` 的前若干个真实 token 输入已完成有限步训练的 `MyGPT`，读取第一个 GPT Block 的 `attention` 张量。横轴为被读取的 key 位置，纵轴为发起查询的 query 位置，每个面板对应一个注意力头。

运行前应明确以下形状与数值不变量：

- 权重形状为 `[B, heads, T, T]`；本图固定 `B=1`，两个 `T` 分别对应 query 与 key 位置。
- softmax 后权重非负，每个 `[batch, head, query]` 行沿 key 维求和应为 1。
- 所有 `key_position > query_position` 的上三角未来位置应为 0；该约束对每个 head 都成立。代码以 `1e-7` 检查未来权重、以 `1e-6` 检查行和误差，前者接近理论精确零，后者只吸收浮点 softmax 的舍入误差。
- 因果约束只规定哪些位置不可读取，并不规定允许区域内必须关注哪个 token。


In [ ]:
# 从训练后的首个 GPT Block 提取真实注意力权重，并核对因果性。
import matplotlib.pyplot as plt

ATTENTION_VIS_LENGTH = min(8, input_ids.size(1))
attention_input_ids = input_ids[:1, :ATTENTION_VIS_LENGTH]
attention_positions = torch.arange(ATTENTION_VIS_LENGTH, device=DEVICE)
attention_hidden = (
    model.token_embedding(attention_input_ids)
    + model.position_embedding(attention_positions)
)
first_block = model.blocks[0]
with torch.no_grad():
    _, causal_attention = first_block.attention(
        first_block.norm1(attention_hidden), return_attention=True
    )

expected_shape = (1, first_block.attention.num_heads, ATTENTION_VIS_LENGTH, ATTENTION_VIS_LENGTH)
if tuple(causal_attention.shape) != expected_shape:
    raise RuntimeError(
        f"注意力形状不符合 [B, heads, T, T] 契约：{tuple(causal_attention.shape)}"
    )
attention_for_plot = causal_attention[0].detach().cpu()
row_sum_error = (attention_for_plot.sum(dim=-1) - 1.0).abs().max().item()
future_mask = torch.triu(
    torch.ones(ATTENTION_VIS_LENGTH, ATTENTION_VIS_LENGTH, dtype=torch.bool),
    diagonal=1,
)
future_max = attention_for_plot[:, future_mask].abs().max().item()
minimum_weight = attention_for_plot.min().item()
if row_sum_error > 1e-6 or future_max > 1e-7 or minimum_weight < 0.0:
    raise RuntimeError(
        "注意力权重未满足非负、按行归一化或未来位置为零的数值契约"
    )

attention_token_labels = [
    itos[int(token_id)] for token_id in attention_input_ids[0].detach().cpu()
]
head_count = attention_for_plot.size(0)
column_count = min(4, head_count)
row_count = math.ceil(head_count / column_count)
fig, axes = plt.subplots(
    row_count, column_count,
    figsize=(3.6 * column_count, 3.5 * row_count),
    squeeze=False, constrained_layout=True,
)
for head_index, ax in enumerate(axes.ravel()):
    if head_index >= head_count:
        ax.set_visible(False)
        continue
    image = ax.imshow(
        attention_for_plot[head_index], cmap="viridis", vmin=0.0, vmax=1.0,
        interpolation="nearest", aspect="equal",
    )
    ax.set_title(f"Head {head_index}")
    ax.set_xticks(range(ATTENTION_VIS_LENGTH), attention_token_labels, rotation=45)
    ax.set_yticks(range(ATTENTION_VIS_LENGTH), attention_token_labels)
    ax.set_xlabel("Key：被读取位置")
    ax.set_ylabel("Query：当前位置")
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.78, label="注意力权重")
fig.suptitle("训练后首层因果自注意力：未来区域必须保持为零")
plt.show()

print(
    "注意力契约：",
    {
        "shape": tuple(causal_attention.shape),
        "max_row_sum_error": row_sum_error,
        "max_future_weight": future_max,
        "min_weight": minimum_weight,
    },
)


**应观察到的结论。** 每个面板的上三角未来区域均为零，非零权重只出现在主对角线及其下方；同一 query 行在允许区域内的权重和为 1。不同 head 可以形成不同的历史分配，这不改变共同的因果边界。

**不可误读的边界。** 这组权重来自重复短语料上的微型模型，只用于验证 mask 与张量语义，不能据此推断通用语言关系。注意力权重也不等同于特征重要性、因果贡献或完整解释；残差、MLP、后续层和输出投影仍会改变最终预测。热力图证明的是“未来信息不可读取”，不是模型已经学会正确生成。


### 4.3．Next-token 概率与序列解码：temperature、top-k 与 beam search

**学习问题。** 对同一个 Prompt、同一模型和同一份最后位置 logits，仅改变 temperature 与 top-k 时，概率分布如何变化？Beam Search 如何从单步概率构造序列候选？为什么某些解码配置会放大重复退化？

对词表大小为 $V$ 的最后位置 logits $\mathbf{z}\in\mathbb{R}^{V}$，Temperature（温度）$T>0$ 通过 $p_i(T)=\operatorname{softmax}(z_i/T)$ 得到下一 Token 的概率。$T<1$ 会放大 logit 差距，使概率更集中；$T>1$ 会压缩差距，使概率更平缓；$T\to0^+$ 时趋近于 Argmax。正温度不改变候选排序，它只改变候选之间的相对概率。Temperature 是逐步作用于条件分布的解码参数，不会改变模型参数，也不能补充模型没有学到的知识。

下列 small multiples（小多图）固定 `prompt` 产生的真实 `next_token_logits`。三列依次改变 temperature，第一行保留完整词表，第二行应用与 `my_generate()` 相同的 top-k 阈值策略。每个面板使用相同纵轴，因此柱高可以直接比较。

运行前应明确以下形状与数值不变量：

- 原始 logits 与每组概率的形状都为 `[V]`，其中 `V=vocab_size`；解码策略不改变词表维度。
- 每组概率均非负且总和为 1。正 temperature 只缩放 logits，不改变其排序。
- top-k 将阈值之外的 logits 置为负无穷，对应概率严格为 0，再对保留候选重新归一化；阈值处并列时可能保留多于 `k` 个候选。
- 所有面板复用完全相同的模型输出，因此差异只来自解码变换，不来自重新训练或再次前向计算。


In [ ]:
# 固定一份真实 next-token logits，仅改变 temperature 与 top-k。
@torch.no_grad()
def my_sampling_probabilities(logits, temperature, top_k=None):
    """把固定 logits 按 temperature 和可选 top-k 转为归一化概率。"""
    if temperature <= 0.0:
        raise ValueError("temperature 必须为正数")
    scaled_logits = logits.clone() / temperature
    retained = torch.ones_like(scaled_logits, dtype=torch.bool)
    if top_k is not None:
        if top_k < 1:
            raise ValueError("top_k 至少为 1")
        k = min(int(top_k), scaled_logits.numel())
        threshold = torch.topk(scaled_logits, k).values[-1]
        retained = scaled_logits >= threshold
        scaled_logits = scaled_logits.masked_fill(~retained, float("-inf"))
    probabilities = F.softmax(scaled_logits, dim=-1)
    return probabilities, retained

with torch.no_grad():
    probability_context = prompt[:, -model.block_size:]
    next_token_logits = (
        model(probability_context)["logits"][0, -1].detach().cpu()
    )

# 0.5/0.8/1.2 仅形成低、中、高温度对照；模型或词表变化后需重新评测质量与多样性。
temperature_values = (0.5, 0.8, 1.2)
policy_specs = [
    (temperature, top_k)
    for top_k in (None, 5)
    for temperature in temperature_values
]
policy_results = []
base_ranking = torch.argsort(next_token_logits, descending=True)
for temperature, top_k in policy_specs:
    probabilities, retained = my_sampling_probabilities(
        next_token_logits, temperature=temperature, top_k=top_k
    )
    if tuple(probabilities.shape) != (vocab_size,):
        raise RuntimeError("解码策略改变了词表维形状")
    if abs(probabilities.sum().item() - 1.0) > 1e-6 or probabilities.min().item() < 0.0:
        raise RuntimeError("next-token 概率未满足非负或归一化契约")
    if not torch.equal(
        torch.argsort(next_token_logits / temperature, descending=True), base_ranking
    ):
        raise RuntimeError("正 temperature 不应改变 logits 排序")
    if top_k is not None and torch.count_nonzero(probabilities[~retained]).item() != 0:
        raise RuntimeError("top-k 阈值之外仍存在非零概率")
    positive_probabilities = probabilities[probabilities > 0]
    entropy = -(positive_probabilities * positive_probabilities.log()).sum().item()
    policy_results.append((temperature, top_k, probabilities, retained, entropy))

vocab_labels = [itos[index] for index in range(vocab_size)]
common_y_max = min(1.0, 1.12 * max(
    probabilities.max().item()
    for _, _, probabilities, _, _ in policy_results
))
fig, axes = plt.subplots(
    2, 3, figsize=(16, 8), sharex=True, sharey=True, constrained_layout=True
)
for ax, (temperature, top_k, probabilities, retained, entropy) in zip(
    axes.ravel(), policy_results
):
    top_token_id = int(probabilities.argmax())
    colors = [
        "#f59e0b" if token_id == top_token_id else "#2563eb"
        for token_id in range(vocab_size)
    ]
    ax.bar(range(vocab_size), probabilities.numpy(), color=colors, width=0.82)
    top_k_label = "完整词表" if top_k is None else str(top_k)
    ax.set_title(f"temperature={temperature}｜top-k={top_k_label}")
    ax.set_ylim(0.0, common_y_max)
    ax.set_xticks(range(vocab_size), vocab_labels, rotation=60, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.text(
        0.98, 0.92,
        f"entropy={entropy:.3f}\nsupport={int(retained.sum())}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
    )
for ax in axes[:, 0]:
    ax.set_ylabel("next-token 概率")
fig.supxlabel("候选 token")
fig.suptitle(
    f"固定 Prompt {my_decode(prompt[0].cpu())!r} 的同一份 logits：解码策略对照"
)
plt.show()

print(
    "next-token 契约：",
    {
        "logits_shape": tuple(next_token_logits.shape),
        "prompt_token_count": probability_context.size(1),
        "argmax_token": itos[int(next_token_logits.argmax())],
        "policies": [
            {
                "temperature": temperature,
                "top_k": top_k,
                "probability_sum": probabilities.sum().item(),
                "support": int(retained.sum()),
                "entropy": entropy,
            }
            for temperature, top_k, probabilities, retained, entropy in policy_results
        ],
    },
)


**应观察到的结论。** 在完整词表一行，较低 temperature 通常使概率集中于高 logit token、熵下降；较高 temperature 使分布更平缓，但候选排序保持不变。应用 top-k 后，尾部候选概率变为 0，保留集合内部重新归一化。由此可区分两个作用：temperature 调整相对尖锐程度，top-k 调整可采样支持集。

#### 4.3.1．Beam Search：从单步概率搜索完整序列

Greedy Search（贪心搜索）在每一步只保留概率最高的一个 Token。Beam Search（束搜索）则维护 $B$ 条候选前缀：每一步将这 $B$ 条前缀分别扩展到词表中的候选 Token，得到至多 $B\times V$ 条新前缀，再按累计分数保留最高的 $B$ 条。基本序列分数为 $s(\mathbf{y}_{1:t})=\sum_{j=1}^{t}\log p(y_j\mid\mathbf{y}_{<j},\mathbf{x})$；生产实现通常还需要长度归一化或长度惩罚，否则短序列可能因累加较少的负对数概率而占优。$B=1$ 时，Beam Search 退化为 Greedy Search。

Beam Search 默认不是随机采样：它搜索累计条件概率较高的若干序列，但每一步都会剪枝，因此不能保证找到整个序列空间的全局最优解。它适合机器翻译等输出约束较强、候选目标相对集中的任务。开放式语言生成通常更重视多样性，较大的 Beam 容易偏向通用、保守或模板化文本，并使每个请求维护的候选状态和 KV Cache 近似随 $B$ 增长。因此生产对话模型通常以受控采样为主，而不是直接增大 Beam。

#### 4.3.2．重复退化：自回归条件分布的正反馈

自回归生成会把本步输出立即追加到下一步输入。如果某个前缀已经出现重复片段，模型的下一步条件分布可能继续偏好该片段；新生成的重复内容又强化最近上下文中的同一模式，使后续概率进一步集中，形成“重复前缀 → 重复续写概率升高 → 新重复进入前缀”的正反馈。训练时的 Next-token Loss 主要约束局部条件预测，并不直接保证完整生成序列没有循环；生成时模型还持续消费自己的输出，因此局部偏差可能跨步累积。

低 Temperature 不是重复的充分条件，但当重复续写已经是最高 logit 候选时，它会压低其他候选的相对概率，减少随机采样跳出循环的机会。Greedy Search 会持续选择这一最高概率路径；Beam Search 也不天然解决重复，因为多个 Beam 可能共享同一重复前缀，累计高概率还可能让重复路径在剪枝中持续保留。缺少 EOS、最大生成长度、重复检测或停止条件时，这种退化会一直持续到外部 Token 预算或上下文边界将其截断。

因此三者的因果关系应明确区分：模型 logits 决定当前偏好，Temperature 重塑单步概率的尖锐程度，Beam Search 管理多条序列候选，而重复退化来自模型输出重新进入条件上下文后的跨步反馈。解码参数只能放大或缓解已有倾向，不能替代模型质量、停止契约和在线循环检测。

**不可误读的边界。** 柱状图是微型模型对单个 Prompt 的条件分布，不代表跨样本质量、事实置信度或概率校准。temperature 与 top-k 只改变解码策略，不能补充模型未学到的知识，也不能保证生成结果正确；一次采样得到的 token 仍受随机数状态影响。top-k 阈值并列可能保留多于 `k` 个 token，这是本实现显式保留并在图中报告 `support` 的边界。本节说明的是模型解码退化；服务端重复拼接流式分片、错误复用 KV Cache 或应用层反复重试属于工程故障，需要在部署链路中单独诊断。


## 5．迁移到生产库

### 5.1．Hugging Face Transformers 接口

本节创建随机初始化的小模型，验证 `input_ids → logits/loss → generate` 的标准契约。原理实现 `MyGPT` 接收已经右移一位的 labels；Transformers Causal LM 约定 `labels=input_ids`，并在模型内部完成 logits/labels 移位。两种协议分别在不同边界执行，已经右移的 labels 不再交给会内部移位的模型。真实预训练权重还需记录模型 ID、Tokenizer 配置和权重哈希。

<!-- diagram:gpt-generation-cache -->
推理阶段先对 Prompt 做一次 Prefill，随后每步只追加一个 token，并复用 KV Cache：

```mermaid
flowchart LR
    P["Prompt tokens"] --> F["Prefill"]
    F --> K["KV Cache"]
    F --> L["最后位置 logits"]
    L --> S["采样或贪心选择"]
    S --> N["新 token"]
    N --> D["单步 Decode"]
    K --> D
    D --> K
    D --> L
```


In [ ]:
# 用 GPT2LMHeadModel 复现相同的 logits、loss 与 generate 接口。

from transformers import GPT2Config, GPT2LMHeadModel

# 复用 64 维、4 头、2 层与四倍 MLP；Dropout 设为 0 以形成可复核的接口对照。
# 集中定义结构和运行参数，避免配置散落在后续逻辑中。
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_embd=64,
    n_layer=2,
    n_head=4,
    n_inner=4 * 64,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    layer_norm_epsilon=1e-5,
    bos_token_id=bos_token_id,
    eos_token_id=eos_token_id,
)
# 实例化当前阶段的模型结构，并准备进入训练或推理模式。
model = GPT2LMHeadModel(config).to(DEVICE)
# GPT2LMHeadModel 在内部完成 next-token shift，因此 labels 与 input_ids 相同。
outputs = model(input_ids=input_ids, labels=input_ids)
print("Transformers output loss:", float(outputs.loss.detach().cpu()))


## 6．生产边界

### 6.1．BERT 与 GPT 的关键差异

| 维度 | BERT | GPT / Llama / Qwen |
|---|---|---|
| 主体 | Encoder-only | Decoder-only |
| 注意力 | 双向 | 因果单向 |
| 常见目标 | Masked Language Model（掩码语言模型） | Causal Language Model（因果语言模型） |
| 典型输出 | 表征、分类 | 逐 token 生成 |

### 6.2．生产验收

- 数据必须包含 tokenizer、模板和样本版本，避免训练/推理模板漂移；
- 长序列需要明确 RoPE（Rotary Position Embedding，旋转位置编码）或其他位置方案；
- 推理要验证 KV Cache、停止条件、采样参数和批量一致性；
- 评估至少包含 loss/perplexity、任务指标、安全集和真实流量延迟；
- 本例是原理实现，不应用于生产服务；生产训练应使用成熟模型实现、分布式策略和可恢复检查点。
